In [0]:
from pyspark.sql import functions as F

storage_account = "stlakehouseds"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/amazon_delta"
gold_base   = f"abfss://gold@{storage_account}.dfs.core.windows.net"

silver = spark.read.format("delta").load(silver_path)
print("Lignes silver :", silver.count())

Lignes silver : 1465


indicateurs par catégorie

In [0]:
kpi_categorie = (silver
    .groupBy("main_category")
    .agg(
        F.countDistinct("product_id").alias("nb_produits"),
        F.round(F.avg("rating"), 2).alias("note_moyenne"),
        F.round(F.avg("discount_percentage"), 1).alias("remise_moyenne_pct"),
        F.round(F.avg("discounted_price"), 2).alias("prix_moyen_remise"),
        F.sum("rating_count").alias("total_avis")
    )
    .orderBy(F.desc("nb_produits")))

display(kpi_categorie)

(kpi_categorie.write
    .format("delta").mode("overwrite")
    .save(f"{gold_base}/kpi_par_categorie"))

main_category,nb_produits,note_moyenne,remise_moyenne_pct,prix_moyen_remise,total_avis
Electronics,490,4.08,50.8,5965.89,15778848
Home&Kitchen,448,4.04,40.1,2330.62,2991069
Computers&Accessories,375,4.15,54.0,842.65,7728689
OfficeProducts,31,4.31,12.4,301.58,149675
HomeImprovement,2,4.25,57.5,337.0,8566
MusicalInstruments,2,3.9,46.0,638.0,88882
Car&Motorbike,1,3.8,42.0,2339.0,1118
Toys&Games,1,4.3,0.0,150.0,15867
Health&PersonalCare,1,4.0,53.0,899.0,3663


top 10 des produits les mieux notés

In [0]:
top_produits = (silver
    .filter(F.col("rating_count") >= 1000)
    .select("product_id", "product_name", "main_category",
            "rating", "rating_count", "discounted_price")
    .orderBy(F.desc("rating"), F.desc("rating_count"))
    .limit(10))

display(top_produits)

(top_produits.write
    .format("delta").mode("overwrite")
    .save(f"{gold_base}/top_produits"))

product_id,product_name,main_category,rating,rating_count,discounted_price
B0BR4F878Q,"Swiffer Instant Electric Water Heater Faucet Tap Home-Kitchen Instantaneous Water Heater Tank less for Tap, LED Electric Head Water Heaters Tail Gallon Comfort(3000W) ((Pack of 1))",Home&Kitchen,4.8,53803,1439.0
B0B53DS4TF,"Instant Pot Air Fryer, Vortex 2QT, Touch Control Panel, 360° EvenCrisp™ Technology, Uses 95 % less Oil, 4-in-1 Appliance: Air Fry, Roast, Bake, Reheat (Vortex 1.97Litre, Black)",Home&Kitchen,4.8,3964,4995.0
B0B23LW7NV,Spigen EZ Fit Tempered Glass Screen Protector for iPhone 14 Pro Max - 2 Pack (Sensor Protection),Electronics,4.7,7779,999.0
B09WN3SRC7,Sony Bravia 164 cm (65 inches) 4K Ultra HD Smart LED Google TV KD-65X74K (Black),Electronics,4.7,5935,77990.0
B0BP89YBC1,Campfire Spring Chef Prolix Instant Portable Water Heater Geyser 1Ltr. for Use Home Stainless Steel Baking Rack | Restaurant | Office | Labs | Clinics | Saloon | with Installation Kit (With MCB),Home&Kitchen,4.7,2591,1499.0
B0BM9H2NY9,Multifunctional 2 in 1 Electric Egg Boiling Steamer Egg Frying Pan Egg Boiler Electric Automatic Off with Egg Boiler Machine Non-Stick Electric Egg Frying Pan-Tiger Woods (Multy),Home&Kitchen,4.7,2300,699.0
B0BM4KTNL1,"FIGMENT Handheld Milk Frother Rechargeable, 3-Speed Electric Frother for Coffee with 2 Whisks and Coffee Decoration Tool, Coffee Frother Mixer, CRESCENT ENTERPRISES VRW0.50BK (A1)",Home&Kitchen,4.7,1729,699.0
B01J1CFO5I,Redgear MP35 Speed-Type Gaming Mousepad (Black/Red),Computers&Accessories,4.6,33434,299.0
B095RTJH1M,Spigen EZ Fit Tempered Glass Screen Protector Guard for iPhone 14/13/13 Pro - 2 Pack,Electronics,4.6,26603,999.0
B01MQ2A86A,"Logitech M331 Silent Plus Wireless Mouse, 2.4GHz with USB Nano Receiver, 1000 DPI Optical Tracking, 3 Buttons, 24 Month Life Battery, PC/Mac/Laptop - Black",Computers&Accessories,4.6,12375,1295.0


top 10 des plus grosses remises


In [0]:
top_remises = (silver
    .select("product_id", "product_name", "main_category",
            "actual_price", "discounted_price", "discount_percentage")
    .orderBy(F.desc("discount_percentage"))
    .limit(10))

display(top_remises)

(top_remises.write
    .format("delta").mode("overwrite")
    .save(f"{gold_base}/top_remises"))

product_id,product_name,main_category,actual_price,discounted_price,discount_percentage
B097C564GC,"rts [2 Pack] Mini USB C Type C Adapter Plug, Type C Female to USB A Male Charger Charging Cable Adapter Converter compatible for iPhone, Samsung S20 ultra/S21/S10/S8/S9/MacBook Pro iPad Silver",Computers&Accessories,4999.0,294.0,94.0
B0BF563HB4,"Fire-Boltt Ninja Call Pro Plus 1.83"" Smart Watch with Bluetooth Calling, AI Voice Assistance, 100 Sports Modes IP67 Rating, 240*280 Pixel High Resolution",Electronics,19999.0,1799.0,91.0
B0BF57RN3K,"Fire-Boltt Ninja Call Pro Plus 1.83"" Smart Watch with Bluetooth Calling, AI Voice Assistance, 100 Sports Modes IP67 Rating, 240*280 Pixel High Resolution",Electronics,19999.0,1799.0,91.0
B0BF54LXW6,"Fire-Boltt Ninja Call Pro Plus 1.83"" Smart Watch with Bluetooth Calling, AI Voice Assistance, 100 Sports Modes IP67 Rating, 240*280 Pixel High Resolution",Electronics,19999.0,1799.0,91.0
B0BF54972T,"Fire-Boltt Ninja Call Pro Plus 1.83"" Smart Watch with Bluetooth Calling, AI Voice Assistance, 100 Sports Modes IP67 Rating, 240*280 Pixel High Resolution",Electronics,19999.0,1799.0,91.0
B0BF4YBLPX,"Fire-Boltt Ninja Call Pro Plus 1.83"" Smart Watch with Bluetooth Calling, AI Voice Assistance, 100 Sports Modes IP67 Rating, 240*280 Pixel High Resolution",Electronics,19999.0,1799.0,91.0
B096MSW6CT,"Sounce Fast Phone Charging Cable & Data Sync USB Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini & iOS Devices",Computers&Accessories,1899.0,199.0,90.0
B0B61DSF17,beatXP Kitchen Scale Multipurpose Portable Electronic Digital Weighing Scale | Weight Machine With Back light LCD Display | White |10 kg | 2 Year Warranty |,Home&Kitchen,1999.0,199.0,90.0
B085HY1DGR,Sounce Spiral Charger Cable Protector Data Cable Saver Charging Cord Protective Cable Cover Headphone MacBook Laptop Earphone Cell Phone Set of 3 (Cable Protector (12 Units)),Computers&Accessories,999.0,99.0,90.0
B08W56G1K9,LAPSTER Spiral Charger Spiral Charger Cable Protectors for Wires Data Cable Saver Charging Cord Protective Cable Cover Set of 3 (12 Pieces),Computers&Accessories,999.0,99.0,90.0


indicateurs globaux (une seule ligne)

In [0]:
kpi_global = silver.agg(
    F.countDistinct("product_id").alias("nb_produits"),
    F.countDistinct("main_category").alias("nb_categories"),
    F.round(F.avg("rating"), 2).alias("note_moyenne"),
    F.round(F.avg("discount_percentage"), 1).alias("remise_moyenne_pct"),
    F.round(F.avg("discounted_price"), 2).alias("prix_moyen_remise")
)

display(kpi_global)

(kpi_global.write
    .format("delta").mode("overwrite")
    .save(f"{gold_base}/kpi_global"))

nb_produits,nb_categories,note_moyenne,remise_moyenne_pct,prix_moyen_remise
1351,9,4.1,47.7,3125.31


vérification

In [0]:
for t in ["kpi_par_categorie", "top_produits", "top_remises", "kpi_global"]:
    n = spark.read.format("delta").load(f"{gold_base}/{t}").count()
    print(f"{t} : {n} lignes")

kpi_par_categorie : 9 lignes
top_produits : 10 lignes
top_remises : 10 lignes
kpi_global : 1 lignes


In [0]:
display(spark.sql("SHOW CATALOGS"))
print(spark.sql("SELECT current_catalog()").first()[0])

catalog
databricks_lakehouse
samples
system


databricks_lakehouse


In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
print("Catalogue utilisé :", catalog)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

for t in ["kpi_par_categorie", "top_produits", "top_remises", "kpi_global"]:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.gold.{t}
        USING DELTA
        LOCATION 'abfss://gold@stlakehouseds.dfs.core.windows.net/{t}'
    """)
    print(t, "enregistrée")

Catalogue utilisé : databricks_lakehouse
kpi_par_categorie enregistrée
top_produits enregistrée
top_remises enregistrée
kpi_global enregistrée
